# 06 · Build a RAG Pipeline：跑通 Naive RAG

Naive RAG（最小 RAG）只有三步：**检索 → 拼提示词 → 生成答案**。前面每个动作都已经单独看过，现在把它们连起来。

In [1]:
import os
import re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None

def split_markdown(text, max_chars=800):
    sections = re.split(r"\n(?=#{1,3}\s)", text)
    chunks, current = [], ""
    for section in sections:
        if current and len(current) + len(section) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += section + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for text in split_markdown(path.read_text(encoding="utf-8")):
            items.append({"source": str(path.relative_to(data_dir)), "text": text})
    return items

def embed(texts):
    if not client:
        raise RuntimeError("请先配置 LLM_API_KEY 或 OPENAI_API_KEY。")
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in response.data], dtype=float)

In [2]:
chunks = load_chunks()
chunk_vectors = embed([item["text"] for item in chunks]) if client else None
if chunk_vectors is not None:
    chunk_vectors /= np.linalg.norm(chunk_vectors, axis=1, keepdims=True)

def search(question, top_k=4):
    question_vector = embed([question])[0]
    question_vector /= np.linalg.norm(question_vector)
    scores = chunk_vectors @ question_vector
    indexes = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[index], "score": float(scores[index])} for index in indexes]

## 先看检索结果和最终提示词

In [3]:
question = "SKU-YG301 瑜伽裤的面料成分和防透光要求是什么？"
results = search(question, top_k=4) if client else []
context = "\n\n".join(f"[来源：{item['source']}]\n{item['text']}" for item in results)
prompt = f"""请只根据下面的资料回答问题。
资料没有答案时，请回答“现有资料无法回答”，不要猜测。

资料：
{context}

问题：{question}"""
print(prompt[:2500] if prompt else "请配置 API 后生成检索结果")

请只根据下面的资料回答问题。
资料没有答案时，请回答“现有资料无法回答”，不要猜测。

资料：


问题：SKU-YG301 瑜伽裤的面料成分和防透光要求是什么？


## 生成答案

In [4]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "你是服饰箱包知识库助手，只根据资料回答，并指出依据的来源。"},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print("未调用模型：请配置 API 后重新运行")

未调用模型：请配置 API 后重新运行


到这里，最小 RAG 已经完成。它没有重排、评估或复杂模块，先把核心闭环跑通。